In [ ]:
# ============================================================
#  CELL 1 -- Dataset Loading + Preprocessing + Augmentation
#  Project : RVCE EL -- AMD vs Normal Fundus Classifier (ViT-B/16)
#  Improvements: fundus-specific augmentation, random CLAHE
# ============================================================

import os
import pathlib
import warnings
import math
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')


# -- 1. Config
BASE = (
    '/kaggle/input/datasets/abuhurer'
    '/macular-degeneration-disease-armd'
    '/Macular Degeneration Disease Dataset'
)

CFG = dict(
    TRAIN_AMD      = f'{BASE}/train/amd',
    TRAIN_NORMAL   = f'{BASE}/train/normal',
    VAL_AMD        = f'{BASE}/val/amd',
    VAL_NORMAL     = f'{BASE}/val/normal',
    CLASS_NAMES    = ['Normal', 'AMD'],
    IMG_SIZE       = 224,
    CLAHE_CLIP_LO  = 1.5,
    CLAHE_CLIP_HI  = 3.0,
    CLAHE_TILE     = (8, 8),
    BATCH_SIZE     = 32,
    NUM_WORKERS    = 4,
    SEED           = 42,
)

torch.manual_seed(CFG['SEED'])
np.random.seed(CFG['SEED'])


# -- 2. Sanity check
print('\n[Sanity] Checking dataset mount ...')
for key in ('TRAIN_AMD', 'TRAIN_NORMAL', 'VAL_AMD', 'VAL_NORMAL'):
    d = pathlib.Path(CFG[key])
    if not d.exists():
        raise FileNotFoundError(f'Directory not found: {d}')
    files = list(d.iterdir())
    print(f'  {key:15s} {len(files):4d} files  |  sample: {files[0].name}')


# -- 3. Preprocessing helpers
def apply_clahe(bgr_img, clip=2.0):
    gray     = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=clip, tileGridSize=CFG['CLAHE_TILE'])
    enhanced = clahe.apply(gray)
    return np.stack([enhanced, enhanced, enhanced], axis=-1)

def center_crop(img, crop_size):
    h, w = img.shape[:2]
    if h < crop_size or w < crop_size:
        pad_h = max(0, crop_size - h)
        pad_w = max(0, crop_size - w)
        img = cv2.copyMakeBorder(img,
            pad_h // 2, pad_h - pad_h // 2,
            pad_w // 2, pad_w - pad_w // 2,
            cv2.BORDER_REFLECT_101)
        h, w = img.shape[:2]
    top  = (h - crop_size) // 2
    left = (w - crop_size) // 2
    return img[top : top + crop_size, left : left + crop_size]

def load_and_preprocess(path, clahe_clip=2.0):
    img = cv2.imread(path)
    img = apply_clahe(img, clahe_clip)
    img = center_crop(img, CFG['IMG_SIZE'])
    img = cv2.resize(img, (CFG['IMG_SIZE'], CFG['IMG_SIZE']), interpolation=cv2.INTER_AREA)
    return img


# -- 4. Augmentation pipelines
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    # Geometric
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=20, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=0,
                       border_mode=cv2.BORDER_REFLECT_101, p=0.4),
    # Fundus-specific
    A.GridDistortion(num_steps=5, distort_limit=0.15,
                     border_mode=cv2.BORDER_REFLECT_101, p=0.3),
    A.RandomGamma(gamma_limit=(80, 120), p=0.4),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.4),
    # Intensity
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.20, p=0.5),
    # Noise / blur
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.MedianBlur(blur_limit=3),
    ], p=0.25),
    A.GaussNoise(var_limit=(5.0, 25.0), p=0.25),
    # Occlusion robustness
    A.CoarseDropout(max_holes=6, max_height=24, max_width=24, fill_value=0, p=0.20),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


# -- 5. Dataset
class FundusDataset(Dataset):
    EXTS = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')

    def __init__(self, amd_dir, normal_dir, transform, random_clahe=False):
        self.transform    = transform
        self.random_clahe = random_clahe
        self.paths, self.labels = [], []
        for ext in self.EXTS:
            for p in pathlib.Path(normal_dir).glob(ext):
                self.paths.append(str(p)); self.labels.append(0)
            for p in pathlib.Path(amd_dir).glob(ext):
                self.paths.append(str(p)); self.labels.append(1)
        print(f'  Normal: {self.labels.count(0):4d}  AMD: {self.labels.count(1):4d}  '
              f'Total: {len(self.paths):4d}')

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        clip = float(np.random.uniform(CFG['CLAHE_CLIP_LO'], CFG['CLAHE_CLIP_HI'])) \
               if self.random_clahe else 2.0
        img   = load_and_preprocess(self.paths[idx], clahe_clip=clip)
        img   = self.transform(image=img)['image']
        label = self.labels[idx]
        return img, torch.tensor(label, dtype=torch.float32)


# -- 6. Build datasets + loaders
print('\n[Train]')
train_dataset = FundusDataset(CFG['TRAIN_AMD'], CFG['TRAIN_NORMAL'],
                               train_transform, random_clahe=True)
print('\n[Val]')
val_dataset   = FundusDataset(CFG['VAL_AMD'],   CFG['VAL_NORMAL'],
                               val_transform,   random_clahe=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'],
                          shuffle=True,  num_workers=CFG['NUM_WORKERS'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG['BATCH_SIZE'],
                          shuffle=False, num_workers=CFG['NUM_WORKERS'], pin_memory=True)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')

imgs, lbls = next(iter(train_loader))
print(f'\n[Sanity] shape={imgs.shape}  dtype={imgs.dtype}  '
      f'range=[{imgs.min():.3f},{imgs.max():.3f}]  '
      f'AMD={int(lbls.sum())}  Normal={int((lbls==0).sum())}')


In [ ]:
# ============================================================
#  CELL 2 -- Model + Two-Phase Training
#  Improvements:
#    - Improved head (LayerNorm + wider MLP)
#    - Phase 1: head-only warm-up  (5 epochs)
#    - Phase 2: full fine-tune with discriminative LRs  (30 epochs)
#    - Focal loss with label smoothing
#    - Linear warm-up + cosine LR schedule
#    - Mixed precision (AMP)
#    - Gradient clipping
#    - Early stopping (patience=7)
#    - Full metrics saved in checkpoint
# ============================================================

import timm
import torch
import torch.nn as nn
import math
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'timm : {timm.__version__}   device : {DEVICE}')

HP = dict(
    PHASE1_EPOCHS = 5,
    PHASE2_EPOCHS = 30,
    HEAD_LR       = 1e-3,
    BACKBONE_LR   = 5e-6,
    WEIGHT_DECAY  = 1e-2,
    GRAD_CLIP     = 1.0,
    PATIENCE      = 7,
    FOCAL_GAMMA   = 2.0,
    FOCAL_ALPHA   = 0.75,
    LABEL_SMOOTH  = 0.05,
    SAVE_PATH     = '/kaggle/working/best_vit_model.pth',
)


# -- Model
class ViTBinaryClassifier(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'vit_base_patch16_224', pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.3),
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(True)


print('\nLoading ViT-B/16 ...')
model = ViTBinaryClassifier(pretrained=True).to(DEVICE)
print(f'Total params : {sum(p.numel() for p in model.parameters()):,}')


# -- Focal Loss
class FocalBCELoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.ls = label_smoothing

    def forward(self, pred, target):
        if self.ls > 0:
            target = target * (1 - self.ls) + 0.5 * self.ls
        eps = 1e-8
        pred = pred.clamp(eps, 1 - eps)
        bce  = -(target * torch.log(pred) + (1 - target) * torch.log(1 - pred))
        p_t  = pred * target + (1 - pred) * (1 - target)
        at   = self.alpha * target + (1 - self.alpha) * (1 - target)
        return (at * (1 - p_t) ** self.gamma * bce).mean()


criterion = FocalBCELoss(HP['FOCAL_ALPHA'], HP['FOCAL_GAMMA'], HP['LABEL_SMOOTH'])
scaler    = GradScaler()


# -- Scheduler
def cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps, min_ratio=0.05):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# -- One epoch
def run_epoch(model, loader, criterion, optimizer, scheduler, is_train):
    model.train(is_train)
    total_loss, n = 0.0, 0
    all_labels, all_preds = [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            with autocast():
                out  = model(imgs).view(-1)
                loss = criterion(out, labels)
            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    HP['GRAD_CLIP'])
                scaler.step(optimizer); scaler.update()
                if scheduler: scheduler.step()
            total_loss += loss.item() * imgs.size(0)
            n          += imgs.size(0)
            preds = (out.detach().float() >= 0.5).long().cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.long().cpu().tolist())
    return dict(
        loss      = total_loss / max(n, 1),
        accuracy  = float(accuracy_score(all_labels, all_preds)),
        precision = float(precision_score(all_labels, all_preds, zero_division=0)),
        recall    = float(recall_score(all_labels, all_preds, zero_division=0)),
        f1_score  = float(f1_score(all_labels, all_preds, zero_division=0)),
    )


# ================================================
# PHASE 1 -- Head warm-up (backbone frozen)
# ================================================
model.freeze_backbone()
print(f'\n--- PHASE 1: Head Warm-Up ({HP["PHASE1_EPOCHS"]} epochs) ---')
print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

opt1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=HP['HEAD_LR'], weight_decay=HP['WEIGHT_DECAY'])
steps_p1 = HP['PHASE1_EPOCHS'] * len(train_loader)
sched1   = cosine_schedule_with_warmup(opt1, steps_p1 // 5, steps_p1)

for epoch in range(1, HP['PHASE1_EPOCHS'] + 1):
    tr = run_epoch(model, train_loader, criterion, opt1, sched1, True)
    vl = run_epoch(model, val_loader,   criterion, None, None,   False)
    print(f'  P1 {epoch:02d}/{HP["PHASE1_EPOCHS"]}  '
          f'tr_loss {tr["loss"]:.4f} tr_acc {tr["accuracy"]:.4f}  '
          f'vl_loss {vl["loss"]:.4f} vl_acc {vl["accuracy"]:.4f} '
          f'f1 {vl["f1_score"]:.4f}')


# ================================================
# PHASE 2 -- Full fine-tuning (discriminative LRs)
# ================================================
model.unfreeze_backbone()
print(f'\n--- PHASE 2: Full Fine-Tuning ({HP["PHASE2_EPOCHS"]} epochs) ---')
print(f'Backbone LR: {HP["BACKBONE_LR"]}   Head LR: {HP["HEAD_LR"]*0.5:.2e}')

opt2 = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': HP['BACKBONE_LR']},
    {'params': model.head.parameters(),     'lr': HP['HEAD_LR'] * 0.5},
], weight_decay=HP['WEIGHT_DECAY'])
steps_p2 = HP['PHASE2_EPOCHS'] * len(train_loader)
sched2   = cosine_schedule_with_warmup(opt2, steps_p2 // 10, steps_p2)

best_acc, best_f1, patience = 0.0, 0.0, HP['PATIENCE']

for epoch in range(1, HP['PHASE2_EPOCHS'] + 1):
    tr = run_epoch(model, train_loader, criterion, opt2, sched2, True)
    vl = run_epoch(model, val_loader,   criterion, None, None,   False)

    improved = (vl['accuracy'] > best_acc or
                (vl['accuracy'] == best_acc and vl['f1_score'] > best_f1))
    tag = ''
    if improved:
        best_acc, best_f1 = vl['accuracy'], vl['f1_score']
        patience = HP['PATIENCE']
        tag = '  saved'
        torch.save({
            'state_dict' : model.state_dict(),
            'model_name' : 'ViT-B16 AMD Classifier (Fine-Tuned)',
            'accuracy'   : vl['accuracy'],
            'precision'  : vl['precision'],
            'recall'     : vl['recall'],
            'f1_score'   : vl['f1_score'],
            'epoch'      : epoch,
        }, HP['SAVE_PATH'])
    else:
        patience -= 1

    print(f'  P2 {epoch:02d}/{HP["PHASE2_EPOCHS"]}  '
          f'tr_loss {tr["loss"]:.4f} tr_acc {tr["accuracy"]:.4f}  '
          f'vl_loss {vl["loss"]:.4f} vl_acc {vl["accuracy"]:.4f} '
          f'prec {vl["precision"]:.4f} rec {vl["recall"]:.4f} '
          f'f1 {vl["f1_score"]:.4f}{tag}')

    if patience == 0:
        print(f'\n  Early stopping (patience={HP["PATIENCE"]}).')
        break

print(f'\nBest val accuracy : {best_acc:.4f}')
print(f'Best val F1       : {best_f1:.4f}')
print(f'Checkpoint        : {HP["SAVE_PATH"]}')
